# Imports

In [19]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import torch
import re
import os
import pandas as pd

# Load Dataset

In [20]:
from datasets import load_dataset
dataset=load_dataset("stanfordnlp/imdb")

In [21]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

# Cleaning func

In [22]:
def clean_text(text):
    # remove html
    text=re.sub(r'<.*?>', " ", text)
    # replace multiple sapce/newlines/tabs
    text=re.sub(r'\s+', ' ', text)
    # remove spaces from beginning and end
    text=text.strip()
    return text

## Apply Cleaning Func

In [23]:
train_df=dataset['train'].to_pandas()
test_df=dataset['test'].to_pandas()

In [24]:
train_df['clean_text']=train_df['text'].apply(clean_text)
test_df['clean_text']=test_df['text'].apply(clean_text)

.apply() runs the function on every row.

# Compare Before vs After

In [25]:
for i in range(3):
    print("Original:")
    print(train_df.loc[i, "text"])

    print("\nCleaned:")
    print(train_df.loc[i, "clean_text"])

    print()

Original:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

Never trust preprocessing blindly.

Always inspect a few examples.

# Verify HTML is Gone

In [26]:
html_count = train_df["clean_text"].str.contains("<br />", regex=False).sum()

print(f"HTML tags remaining: {html_count}")

HTML tags remaining: 0


yea, the cleaning func worked!

# Check Random Reviews

In [27]:
train_df['clean_text'].sample(5,random_state=20)

8879     Dark Rising is your typical bad, obviously qui...
12093    I rented this shortly after renting Ben Stein'...
18594    This movie is horrible- in a 'so bad it's good...
18491    by Dane Youssef "Coonskin" is film, by the one...
5764     This movie contains no humor for anyone who ha...
Name: clean_text, dtype: str

# Vocabulary After Cleaning

In [28]:
from collections import Counter
counter=Counter()

for review in train_df['clean_text']:
    counter.update(review.lower().split())

print(f"Vocabulary size: {len(counter)}")
print("Top 20 words: ")
print(counter.most_common(20))

Vocabulary size: 236042
Top 20 words: 
[('the', 329632), ('a', 160691), ('and', 159309), ('of', 144659), ('to', 134232), ('is', 104251), ('in', 91506), ('i', 74221), ('this', 71985), ('it', 66485), ('that', 66477), ('was', 47053), ('as', 45810), ('for', 43154), ('with', 42903), ('but', 40540), ('on', 31813), ('movie', 30916), ('his', 29117), ('not', 28803)]


# Save the Clean Dataset

In [31]:
os.makedirs("../data/processed", exist_ok=True)

In [32]:
train_df.to_csv("../data/processed/train.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

We are using BERT, therefore, we don't need : lowercasing, stopword removal, stemming/lemmatization. The tokenizer's subword vocab handles casing and morphology — stripping that stuff is a classic-ML-era habit that actively removes signal for transformer models.